In [ ]:
### HERE COMES THE ENITRE ANALYSIS AND THE PLOTS I WILL CREATE
### IT WILL ACCESS THE SAME FUNCTIONS (PARTIALLY) AS THE 4_homologs_analysis_visualization.ipynb

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path for custom src imports later
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Default plot style
# sns.set_theme(style="whitegrid", context="notebook")
# plt.rcParams["figure.dpi"] = 100

In [ ]:
DATA_DIR = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"

df_for_rg = pd.read_parquet(f"{DATA_DIR}/variants_annotated_final.parquet")
print(f"Loaded {len(df_for_rg):,} variant-region assignments")
print(f"Columns ({len(df_for_rg.columns)}): {df_for_rg.columns.tolist()}")

# Also load the regions JSON — useful for context (WT sequences, etc.)
with open(f"{DATA_DIR}/genomic_coords_merged_win5.json") as f:
    regions = json.load(f)
region_by_id = {r["region_id"]: r for r in regions}
print(f"Loaded {len(regions)} regions from JSON")

In [ ]:
df_for_rg

In [ ]:
###### NOW ANALYSIS

In [ ]:
import src.analysis_visualization.region_analysis as ra

fig, stats = ra.plot_variant_density(df_for_rg, dataset="gnomad")
plt.show()

In [ ]:
fig, results = ra.plot_consequence_distributions(df_for_rg, dataset="gnomad", drop_other=True)
plt.show()

In [ ]:
fig, r = ra.plot_median_alphamissense(df_for_rg, dataset="gnomad")
plt.show()

In [ ]:
import src.analysis_visualization.esm_llr as esm_llr

fig, r = esm_llr.plot_median_esm_llr(df_for_rg, dataset="gnomad")
plt.show()

In [ ]:
import src.analysis_visualization.rg_analysis as rga


# df_rg = rga.compute_rg_disruption_columns(df_for_rg, region_by_id)

###### NOT NEEDED ANYMORE NOW PERFOREMD IN 1_gnomAD_processing.ipynb


print("RG hit columns added. Summary:")
print(df_for_rg["hits_rg"].value_counts())

# Run each plot independently
fig_a, r_a = rga.plot_region_length(region_by_id, dataset="gnomad")
plt.show()

fig_b, r_b = rga.plot_n_rg_motifs(region_by_id, dataset="gnomad")
plt.show()

fig_c, r_c = rga.plot_rg_density(region_by_id, dataset="gnomad")
plt.show()

# D1 — density version (replaces the saturating D)
fig_d1, r_d1 = rga.plot_variants_per_rg_by_type(df_for_rg, region_by_id, dataset="gnomad")
plt.show()

# # D2 — AlphaMissense on RG-hitting missense
# fig_d2, r_d2 = rga.plot_median_alphamissense_on_rgs(df_for_rg, dataset="gnomad")
# plt.show()


In [ ]:
fig, r = rga.plot_rg_role_asymmetry(df_for_rg, dataset="gnomad")
plt.show()

In [ ]:
results = rga.plot_rg_change_events_stacked(df_for_rg, region_by_id, dataset="gnomad")

# the single event version has been commmented out in the rg-analysis, because it showed no signifincance in any plot, so here is just the stacked version

In [ ]:
# Make sure df_events is computed (already have compute_rg_change_events from before)
df_events = rga.compute_rg_change_events(df_for_rg, region_by_id)

# Analysis 1
loss_transition_results = rga.plot_rg_loss_transitions(df_events, dataset="gnomad")
plt.show()

# Analysis 2
cluster_results = rga.plot_isolated_vs_clustered_loss(
    df_events, region_by_id,
    window_sizes=[2,4,6],
    dataset="gnomad",
)
plt.show()

In [ ]:
gain_transition_results = rga.plot_rg_gain_transitions(df_events, dataset="gnomad")
plt.show()

In [ ]:
# Build the null once — reused for both plots
from src.analysis_visualization.substitution_matrix_analysis import load_mutation_rates
# null_results = rga.build_enumeration_null(region_by_id, df_for_rg)
rates = load_mutation_rates()  # or load_mutation_rates(path) if non-default
null_results = rga.build_enumeration_null(region_by_id, df_for_rg, rates=rates)
# RG events observed vs expected
rg_comparison = rga.plot_rg_events_observed_vs_expected(
    df_events, null_results, dataset="gnomad",
)
plt.show()

# Consequences observed vs expected
cons_comparison = rga.plot_consequences_observed_vs_expected(
    df_for_rg, null_results, dataset="gnomad",
)
plt.show()

In [ ]:
from src.analysis_visualization.substitution_matrix_analysis import load_mutation_rates

rates = load_mutation_rates()  # or load_mutation_rates(path) if non-default
null_results = rga.build_enumeration_null(region_by_id, df_for_rg, rates=rates)
df_events = rga.compute_rg_change_events(df_for_rg, region_by_id)

rg_results   = rga.plot_rg_events_vs_expected_boxes(df_events,  null_results, dataset="gnomad")
cons_results = rga.plot_consequences_vs_expected_boxes(df_for_rg, null_results, dataset="gnomad")

In [ ]:
# Plot A: per-variant distribution, R/G-affecting only
fig_a, r_a = rga.plot_delta_rg_ratio_per_variant(df_for_rg, region_by_id, dataset="gnomad")
plt.show()

# Plot B: per-region mean across all missense
fig_b, r_b = rga.plot_delta_rg_ratio_per_region(df_for_rg, region_by_id, dataset="gnomad")
plt.show()

In [ ]:
from src.analysis_visualization.substitution_matrix_analysis import load_mutation_rates

# rates = load_mutation_rates()
# null_results = build_enumeration_null(region_by_id, df_observed, rates=rates)
rates = load_mutation_rates()  # or load_mutation_rates(path) if non-default
null_results = rga.build_enumeration_null(region_by_id, df_for_rg, rates=rates)

stats_dict = rga.plot_delta_rg_ratio_vs_expected(df_for_rg, region_by_id, null_results)

In [ ]:
import src.analysis_visualization.rg_analysis as rga

fig, res = rga.plot_delta_n_rg_motifs_per_region(df_for_rg, region_by_id, use_density=False)
# or, if region lengths vary a lot:
fig, res = rga.plot_delta_n_rg_motifs_per_region(df_for_rg, region_by_id, use_density=True)

In [ ]:
from src.analysis_visualization.substitution_matrix_analysis import load_mutation_rates

# rates = load_mutation_rates()
# null_results = build_enumeration_null(region_by_id, df_observed, rates=rates)
rates = load_mutation_rates()  # or load_mutation_rates(path) if non-default
null_results = rga.build_enumeration_null(region_by_id, df_for_rg, rates=rates)
stats_dict = rga.plot_delta_n_rg_motifs_vs_expected(
    df_for_rg, region_by_id, null_results, use_density=True
)

In [ ]:
############### PHYSCHEM

In [ ]:
import src.analysis_visualization.physchem_analysis as pca

# # Step 1: compute per-variant deltas (slow — uses multiprocessing)
# deltas_df = pca.compute_physchem_deltas(df_rg, region_by_id)

# # Save the result — expensive to recompute
# deltas_df.to_parquet(
#     "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed/physchem_deltas.parquet"
# )

deltas_df = pd.read_parquet("/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed/physchem_deltas.parquet"
)

# Step 2: aggregate to per-region means for classifier features
per_region = pca.aggregate_per_region(deltas_df)

# Step 3: plot each feature individually
all_results = pca.plot_all_delta_features(per_region, dataset="gnomad")

# Bonus: per-region WT baseline features (no variants involved — useful for classifier too)
wt_features = pca.compute_wt_physchem(region_by_id)

In [ ]:
################# AMINO ACID SUBSTITUTIONS

In [ ]:
import src.analysis_visualization.substitution_matrix_analysis as smx

raw   = smx.run_substitution_analysis(df_for_rg, dataset="gnomad", min_total=5)
comp  = smx.run_composition_normalized_analysis(df_for_rg, region_by_id, dataset="gnomad")
mut   = smx.run_mutability_normalized_analysis(df_for_rg, region_by_id, dataset="gnomad")

In [ ]:
mut, subs_scores_table = smx.run_mutability_normalized_analysis(df_for_rg, region_by_id, dataset="gnomad", save_table=False, save_table_path="/mnt/d/phd/scripts/16_ev_signature_predictor/data/output/gnomad_mutability_substitution_table.csv")
print(mut)
print(subs_scores_table)
# all points
smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut", show_significance=True, bg_span=2.0)

# # R-highlighted version
# smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut_r", highlight_sources=["R"],
#                          show_significance=True, bg_span=2.0)

# # G-highlighted version
# smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut_g", highlight_sources=["G"],
#                          show_significance=True, bg_span=2.0)

# # Y-highlighted version
# smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut_y", highlight_sources=["Y"],
                        #  show_significance=True, bg_span=2.0)

# a, b = smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut_cgp", highlight_sources=["C", "G", "P"],
#                          show_significance=False, bg_span=2.0)
# a
# smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut_cgp", highlight_sources=["A", "V", "I", "L", "M"],
#                          show_significance=False, bg_span=2.0)

# smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut_cgp", highlight_sources=list("STNQ"),
#                          show_significance=False, bg_span=2.0)

# smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut_cgp", highlight_sources=list("RKH"),
#                          show_significance=False, bg_span=2.0)

# smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut_cgp", highlight_sources=list("DE"),
#                          show_significance=False, bg_span=2.0)

# smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut_cgp", highlight_sources=list("FWY"),
#                          show_significance=False, bg_span=2.0)

# smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut_cgp", highlight_sources=["A"],
#                         show_significance=False, bg_span=2.0)

# smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut_cgp", highlight_sources=["V"],
#                         show_significance=False, bg_span=2.0)
# smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut_cgp", highlight_sources=["I"],
#                         show_significance=False, bg_span=2.0)
# smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut_cgp", highlight_sources=["L"],
#                         show_significance=False, bg_span=2.0)
# smx.plot_obs_exp_scatter(mut, dataset="gnomad_mut_cgp", highlight_sources=["M"],
#                         show_significance=False, bg_span=2.0)

# smx.plot_obs_exp_scatter(mut, dataset="gnomad_raw", n_label=0, show_significance=True) #n_label=200,always_label_sources=["R","G"]
# smx.plot_obs_exp_scatter(raw, dataset="gnomad_mutability", show_significance=True, always_label_sources=["R","G"])

In [ ]:
from src.analysis_visualization.substitution_matrix_analysis import build_score_table, plot_score_heatmap

table, lookup = build_score_table(subs_scores_table, alpha=0.5)
ax = plot_score_heatmap(table, annotate=False)   # 20x20 will be dense; annotate off

In [ ]:
table

In [ ]:
rates = smx.load_mutation_rates()


grp_result = smx.compute_grouped_substitution_matrix(df_for_rg, region_by_id, rates)

mut = smx.run_mutability_normalized_analysis(df_for_rg, region_by_id, dataset="gnomad")

# all points (use as supplement, or with label_all=False)
a, b = smx.plot_obs_exp_scatter_grouped(mut, dataset="gnomad_mut_all",
                                 show_significance=True, bg_span=1.5, label_all=False, n_label=0)
a
# # highlighted, readable main version
# smx.plot_obs_exp_scatter_grouped(mut, dataset="gnomad_mut_pos",
#                                  highlight_sources=["Pos"], show_significance=True, bg_span=1.5)
# smx.plot_obs_exp_scatter_grouped(mut, dataset="gnomad_mut_cgp",
#                                  highlight_sources=["C/G/P"], show_significance=True, bg_span=1.5)
# smx.plot_obs_exp_scatter_grouped(mut, dataset="gnomad_mut_neg",
#                                  highlight_sources=["Neg"], show_significance=True, bg_span=1.5)
# smx.plot_obs_exp_scatter_grouped(mut, dataset="gnomad_mut_polar",
#                                  highlight_sources=["Polar"], show_significance=True, bg_span=1.5)
# smx.plot_obs_exp_scatter_grouped(mut, dataset="gnomad_mut_aromatic",
#                                  highlight_sources=["Aromatic"], show_significance=True, bg_span=1.5)
# smx.plot_obs_exp_scatter_grouped(mut, dataset="gnomad_mut_hydrophobic",
#                                  highlight_sources=["Hydrophobic"], show_significance=True, bg_span=1.5)

In [ ]:


# grouped heatmap (obs/exp, binomial-tested)
grp = smx.run_grouped_substitution_analysis(df_for_rg, region_by_id, dataset="gnomad_mut")

# grouped scatter — reuses the 20x20 mutability result, swappable grouping
mut = smx.run_mutability_normalized_analysis(df_for_rg, region_by_id, dataset="gnomad")
smx.plot_obs_exp_scatter_grouped(mut, dataset="gnomad_mut")
# smx.plot_obs_exp_scatter_grouped(mut, groups=MY_CHARGE_GROUPING, dataset="gnomad_charge")  # brainstorm

In [ ]:
df_for_rg = smx.annotate_cpg_status(df_for_rg, "/mnt/d/phd/scripts/16_ev_signature_predictor/data/refs/GRCh38.primary_assembly.genome.fa")   # once

rates = smx.load_mutation_rates()
allc  = smx.compute_composition_normalized_enrichment(df_for_rg, region_by_id, rates=rates, cpg_filter="all")
nonc  = smx.compute_composition_normalized_enrichment(df_for_rg, region_by_id, rates=rates, cpg_filter="noncpg")
cpg   = smx.compute_composition_normalized_enrichment(df_for_rg, region_by_id, rates=rates, cpg_filter="cpg")

smx.plot_obs_exp_scatter_grouped(allc, dataset="gnomad_allc")   # the decisive one
# smx.plot_obs_exp_scatter_grouped(cpg,  dataset="gnomad_cpg")
# smx.plot_obs_exp_scatter(nonc, dataset="gnomad_noncpg",n_label= 0, show_significance=True, always_label_sources=["R"])
# smx.plot_obs_exp_scatter(cpg, dataset="gnomad_cpg", show_significance=True, always_label_sources=["R","G"])

In [ ]:
import src.analysis_visualization.substitution_matrix_analysis as smx

# All-missense matrix (single call)
enrichment = smx.run_substitution_analysis(
    df_for_rg, dataset="gnomad", min_total=5,
)

# Access individual outputs
log2_or_matrix = enrichment["log2_or"]
fdr_matrix = enrichment["fdr"]
counts_pos = enrichment["counts_pos"]

In [ ]:
import src.analysis_visualization.substitution_matrix_analysis as smx

# Side-by-side rare (AF < 1e-4) vs common (AF >= 1e-3) matrices
results = smx.plot_af_comparison_matrices(
    df_for_rg,
    af_rare_max=1e-5,
    af_common_min=1e-5,
    dataset="gnomad",
)

# Access individual results
rare_enrichment = results["rare"]
common_enrichment = results["common"]

In [ ]:
# %%
%autoreload 2
from src.analysis_visualization.substitution_matrix_analysis import (
    run_marginal_substitution_analysis,
    SNP_REACHABLE,  # optionally inspect the dict
)

# Sanity check the reachability dict
print("R can reach:", SNP_REACHABLE["R"])
print("W can reach:", SNP_REACHABLE["W"])

# # %%
# result_marginal = run_marginal_substitution_analysis(
#     df,
#     dataset="gnomad",
#     min_total=1,
# )


# Supplementary — all tested AAs
result_marginal = run_marginal_substitution_analysis(df, dataset="gnomad")

# Publication figure — significant AAs only, auto-selected
run_marginal_substitution_analysis(
    df, dataset="gnomad",
    sig_only=True,
    save=True,
)

# Or hand-pick specific AAs you want to highlight
run_marginal_substitution_analysis(
    df, dataset="gnomad",
    source_aas=["G", "P", "R"],
    save=True,
)


# # Just the heatmap
# run_marginal_substitution_analysis(df, dataset="gnomad", plot_kind="enrichment_heatmap", save=True)

# # Focused bars for the significant AAs only
# run_marginal_substitution_analysis(
#     df, dataset="gnomad", sig_only=True,
#     plot_kind="enrichment_bars", save=True,
# )

# # Everything
# run_marginal_substitution_analysis(df, dataset="gnomad", plot_kind="all", save=True)

In [ ]:
# Total G→X variants in common matrix, per group
r_common = results["common"]
g_row_pos = r_common["counts_pos"].loc["G"].sum()
g_row_neg = r_common["counts_neg"].loc["G"].sum()
print(f"G→anything: pos = {g_row_pos}, neg = {g_row_neg}")
print(f"G→S fraction: pos = {77/g_row_pos:.3f}, neg = {19/g_row_neg:.3f}")

In [ ]:
# Quick check
gs_pos = df_rg[
    (df_rg["before_aa"] == "G") & 
    (df_rg["after_aa"] == "S") & 
    (df_rg["group"] == "pos") &
    (df_rg["AF_joint"] >= 1e-5) &
    (df_rg["Consequence"].str.contains("missense_variant", na=False))
]
gs_neg = df_rg[
    (df_rg["before_aa"] == "G") & 
    (df_rg["after_aa"] == "S") & 
    (df_rg["group"] == "neg") &
    (df_rg["AF_joint"] >= 1e-5) &
    (df_rg["Consequence"].str.contains("missense_variant", na=False))
]
print("G→S codon distribution in pos:")
print(gs_pos["Codons"].value_counts().head(10))
print("\nG→S codon distribution in neg:")
print(gs_neg["Codons"].value_counts().head(10))

In [ ]:
#### add here the proportion of each amino acid how often it appears in the regions, so that i can maybe find something intersting

In [ ]:
import src.analysis_visualization.codon_usage as cu

df_comp, comp_stats = cu.compute_gc_cpg_by_group(region_by_id)
cu.plot_gc_cpg_by_group(df_comp, comp_stats, dataset="gnomad")

In [ ]:
rates = smx.load_mutation_rates()


merged, flux_stats, match_rate = cu.compute_gc_cpg_flux(
    df_for_rg, region_by_id, rates, cu.enumerate_single_nt_substitutions)
print("context match rate:", match_rate)   # must be ~1.0 to trust ΔCpG
cu.plot_gc_cpg_flux(merged, flux_stats, dataset="gnomad")

In [ ]:
import pandas as pd
mis = df_for_rg[df_for_rg["Consequence"].fillna("").str.contains("missense_variant")]

def diag(v, region_by_id, shift=0):
    rid = v["region_id"]
    if rid not in region_by_id: return None
    dna = region_by_id[rid]["dna"].upper()
    pc = cu._parse_codons(v.get("Codons"))
    if pc is None: return None
    _, _, off, cref, calt = pc
    ci = int(v["protein_position_int"]) - int(v["region_start_aa"]) + shift
    dpos = 3*ci + off
    if not (0 <= dpos < len(dna)): return ("oob", cref, None)
    return (dna[dpos] == cref, cref, dna[dpos])

# how well does each candidate shift match?
for shift in [-1, 0, 1]:
    res = [diag(v, region_by_id, shift) for _, v in mis.iterrows()]
    ok = sum(1 for r in res if r and r[0] is True)
    tot = sum(1 for r in res if r and r[0] in (True, False))
    print(f"shift={shift:+d}: match {ok}/{tot} = {ok/tot:.2%}" if tot else f"shift={shift}: no valid")

In [ ]:
import src.analysis_visualization.codon_usage as cu

rates = smx.load_mutation_rates()
df_cm, cm_stats, cm = cu.compute_codon_mutability_by_group(
    region_by_id, rates)  
# cu.plot_codon_mutability(df_cm, cm_stats, cm, amino_acids=list("ADEGL"), dataset="gnomad")
# cu.plot_codon_mutability(df_cm, cm_stats, cm, amino_acids=list("PQRSY"), dataset="gnomad")
# cu.plot_codon_mutability(df_cm, cm_stats, cm, amino_acids=list("AVILM"), dataset="gnomad")
# cu.plot_codon_mutability(df_cm, cm_stats, cm, amino_acids=list("CGP"), dataset="gnomad")
cu.plot_codon_mutability(df_cm, cm_stats, cm, amino_acids=list("ADEGLPRS"), dataset="gnomad")
# cu.plot_codon_mutability(df_cm, cm_stats, cm, amino_acids=["R", "G", "P", "C"], dataset="gnomad")

In [ ]:
import src.analysis_visualization.codon_usage as cu

# Supplementary — full grid, all 18 AAs
out = cu.run_codon_usage_analysis(region_by_id, dataset="gnomad")

# Publication — only AAs you want to feature
# cu.plot_codon_usage(
#     out["codon_counts"], out["test_results"],
#     source_aas=list("ADEGLPQRSY"), ncols= 5,
#     dataset="gnomad",
# )
# Publication — only AAs you want to feature
tmp = cu.plot_codon_usage(
    out["codon_counts"], out["test_results"],
    source_aas=list("ADEGLPRS"), ncols= 3,
    dataset="gnomad",
)
# tmp = cu.plot_codon_usage(
#     out["codon_counts"], out["test_results"],
#     source_aas=list("GRPSA"), ncols= 1,
#     dataset="gnomad",
# )


# Auto-select significant
tmp =cu.plot_codon_usage(
    out["codon_counts"], out["test_results"],
    sig_only=False,
    dataset="gnomad",
)

# print('hello')


In [ ]:
#### AF analysis

In [ ]:
import src.analysis_visualization.af_spectrum as afs

stats = afs.plot_af_spectrum_cdf(df_rg, dataset="gnomad")

In [ ]:
import src.analysis_visualization.af_spectrum as afs

# df_rg must have `is_rg_disrupting` column (from compute_rg_disruption_columns)
stats = afs.plot_af_spectrum_by_subset(df_rg, dataset="gnomad")

In [ ]:
# Quick check in notebook
rg = df_rg[df_rg["is_rg_disrupting"] & df_rg["AF_joint"].notna()]
for group in ["pos", "neg"]:
    g = rg[rg["group"] == group]["AF_joint"]
    common = g[g >= 1e-4]
    very_common = g[g >= 1e-3]
    print(f"{group}: n={len(g):,}, n≥1e-4={len(common)}, n≥1e-3={len(very_common)}, "
          f"max={g.max():.4f}")

In [ ]:
##### HERE SWITCH TO 3_RF_model_creation.ipynb